# World session 2 · Dates that lie

**Goal:** read every date the way it was written, then catch the ones that can't be true. Course 1's session 3 read one column of dates. Here, 900,000 charges were filed by many clerks, and a date can be perfectly readable and still wrong: a charge filed before the fight it's about.

**How to use this notebook**
- Click a gray code cell and press **Shift + Enter** to run it. Go top to bottom.
- 🧪 **Your turn** cells contain a `___` blank. Replace it with your answer and run the cell; it tells you whether you're right. Answers are at the bottom if you're stuck.
- Your answers are checked against the world's **answer keys**: the generator's own record of every mess it made. A result you hand in (rows you found) is graded by **precision** (how many of yours are right) and **recall** (how many of the right ones you found).
- Nothing here can change the lake. You read the messy files as a source system delivered them.

> ⚠️ **Mock data only.** Every person, place and record in this world is invented.

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and opens the world's messy files.
SESSION = "2"

import sys
from pathlib import Path

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "lakehouse" / "sessions" / "check.py").exists():
        sys.path.insert(0, str(folder))
        break

import re
import pandas as pd
from lakehouse.sessions import check
from lakehouse.sessions.check import ___

con = check.world()       # DuckDB: one view messy_<table> per messy CSV folder, every column as text
print(f"World session {SESSION} ready. Messy tables:", ", ".join(r[0] for r in con.sql("SHOW TABLES").fetchall()))

## 1. Profile the shapes
Turn every digit into `9` and every letter into `A`. Each shape is one format to handle.

In [ ]:
con.execute("CREATE OR REPLACE MACRO shape(s) AS regexp_replace(regexp_replace(s, '[0-9]', '9', 'g'), '[A-Za-z]', 'A', 'g')")
con.sql("SELECT shape(filed_on) AS shape, min(filed_on) AS example, count(*) AS n FROM messy_charge GROUP BY 1 ORDER BY n DESC").df()

ISO dates, US dates, dates with a time, `01-APR-1996`, month names, and **two-digit years** (`3/10/09`). Then `UNK`, `UNKNOWN`, `?` and `00/00/0000`: a date that was never recorded.

## 2. A first reader
`try_strptime` returns null instead of failing, so `COALESCE` can try formats in turn. This first version knows the four most common ones.

In [ ]:
con.execute("""
CREATE OR REPLACE MACRO read_date_v1(s) AS COALESCE(
    try_strptime(trim(s), '%Y-%m-%d'), try_strptime(trim(s), '%m/%d/%Y'),
    try_strptime(trim(s), '%Y%m%d'), try_strptime(trim(s), '%Y-%m-%d %H:%M:%S'))::DATE
""")
con.sql("SELECT shape(filed_on) AS shape, min(filed_on) AS example, count(*) AS n FROM messy_charge WHERE read_date_v1(filed_on) IS NULL GROUP BY 1 ORDER BY n DESC").df()

What's left: four more formats, the two-digit years, and the stand-ins.

**Two-digit years need a rule.** `3/10/09` could be 1909 or 2009. These records run from 1996 to today (2026), so a two-digit year up to 26 is 20xx and anything higher is 19xx. Write the rule down: it's a decision, not a fact. DuckDB's own `%y` guesses differently, so read those by hand.

🧪 **Your turn #1.** With a full reader (the answer to #2 builds one), how many filing dates can't be read at all, because a stand-in is written where the date should be? For now, count the stand-ins: `UNK`, `UNKNOWN`, `?` and `00/00/0000`.

In [ ]:
answer_1 = ___      # count filed_on values that are one of the four stand-ins
check.w2_ex1(answer_1)

🧪 **Your turn #2.** Write `read_date(s)`, a reader for every format, with the two-digit-year rule, and build `dates`: one row per charge you could read, with `charge_id` and `filed_on` (a date). The bar is 99.9%: every date read exactly as it was written.

In [ ]:
dates = ___      # define read_date, then SELECT charge_id and read_date(filed_on) AS filed_on
check.w2_ex2(dates)

## 3. Readable, and still wrong
A charge is filed after the fight it's about. Read each fight's start time (`messy_incident.started_at` has its own formats, 12-hour times among them) and compare.

In [ ]:
con.sql("SELECT shape(started_at) AS shape, min(started_at) AS example, count(*) AS n FROM messy_incident GROUP BY 1 ORDER BY n DESC").df()

In [ ]:
con.execute("""
CREATE OR REPLACE MACRO read_time(s) AS COALESCE(
    try_strptime(trim(s), '%Y-%m-%d %H:%M:%S'), try_strptime(trim(s), '%Y-%m-%dT%H:%M:%S'),
    try_strptime(trim(s), '%d-%b-%Y %H:%M:%S'), try_strptime(trim(s), '%m/%d/%Y %I:%M:%S %p'),
    try_strptime(trim(s), '%m/%d/%Y %H:%M:%S'), try_strptime(trim(s), '%Y%m%d%H%M%S'))
""")
con.sql("SELECT started_at, read_time(started_at) FROM messy_incident USING SAMPLE 5 ROWS (reservoir, 7)").df()

🧪 **Your turn #3.** Build `before_the_fight`: the `charge_id` of every charge whose filing date is before the date its fight started. Use your `read_date` from #2. The bar is 99% precision and 95% recall (a few hide behind a date nobody recorded).

In [ ]:
before_the_fight = ___      # charge_ids where read_date(filed_on) < the fight's date
check.w2_ex3(before_the_fight)

## 4. How the lake does it
Silver reads every date this way and never drops a row. A date it can't read becomes null with a flag; a date out of order keeps its value and gets a flag, `out_of_order`, in `world_silver.exceptions`. The decision about what's true stays with the people who know.

In [ ]:
from lakehouse import lake
silver = lake.lake_ro_connect()
silver.sql("SELECT rule, count(*) FROM lake.world_silver.exceptions WHERE table_name = 'charge' AND column_name = 'filed_on' GROUP BY 1 ORDER BY 2 DESC").df()

## Wrap-up
- Profile shapes before writing a parser; each shape is a format or a stand-in.
- Two-digit years need a written rule (here: up to 26 is 20xx).
- A stand-in is unknown, not a date: it reads as null.
- A readable date can still break a rule (filed before the fight). Flag it and keep the row; don't swap in a guess.

## Answers
Try each exercise first. Each answer replaces the `___` line in its 🧪 cell.

**#1** Count the four stand-ins as written.

```python
answer_1 = con.sql("""
    SELECT count(*) FROM messy_charge
    WHERE upper(trim(filed_on)) IN ('UNK', 'UNKNOWN', '?', '00/00/0000')
""").fetchone()[0]
```

**#2** Every format from the profile, two-digit years by the rule, then read the column.

```python
dates = con.execute("""
CREATE OR REPLACE MACRO read_date(s) AS CASE
    WHEN regexp_matches(trim(s), '^\\d{1,2}/\\d{1,2}/\\d{2}$') THEN          -- 3/22/01: a two-digit year
        make_date(CASE WHEN CAST(split_part(trim(s), '/', 3) AS INT) <= 26 THEN 2000 ELSE 1900 END
                      + CAST(split_part(trim(s), '/', 3) AS INT),
                  CAST(split_part(trim(s), '/', 1) AS INT), CAST(split_part(trim(s), '/', 2) AS INT))
    ELSE COALESCE(try_strptime(trim(s), '%Y-%m-%d'),          -- 2001-03-22
                  try_strptime(trim(s), '%Y-%m-%d %H:%M:%S'), -- 2001-03-22 00:00:00
                  try_strptime(trim(s), '%m/%d/%Y'),          -- 03/22/2001
                  try_strptime(trim(s), '%m-%d-%Y'),          -- 03-22-2001
                  try_strptime(trim(s), '%Y%m%d'),            -- 20010322
                  try_strptime(trim(s), '%d-%b-%Y'),          -- 22-MAR-2001
                  try_strptime(trim(s), '%b %d %Y'),          -- Mar 22 2001
                  try_strptime(trim(s), '%B %d, %Y'))::DATE   -- March 22, 2001
END
""").sql("""
    SELECT CAST(charge_id AS BIGINT) AS charge_id, read_date(filed_on) AS filed_on
    FROM messy_charge WHERE read_date(filed_on) IS NOT NULL
""").df()
```

**#3** Join each charge to its fight and compare the dates.

```python
before_the_fight = con.sql("""
    SELECT CAST(c.charge_id AS BIGINT) AS charge_id
    FROM messy_charge c
    JOIN messy_incident i ON i.incident_id = c.incident_id
    WHERE read_date(c.filed_on) < CAST(read_time(i.started_at) AS DATE)
""").df()
```